In [ ]:
import os
from pathlib import Path
import csv

import torch
from PIL import Image
import pandas as pd
from huggingface_hub import hf_hub_download

from open_flamingo import create_model_and_transforms  # from med-flamingo/open_flamingo

import re


In [ ]:

# ---------- CONFIG ----------
MODEL_REPO = "med-flamingo/med-flamingo"  # HF repo
CHECKPOINT_FILENAME = "model.pt"          # in that repo
LLAMA_PATH = "openlm-research/open_llama_7b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Put your folder + prompt here
INPUT_DIR = "./data/LLaVA-Med/images/test"
OUTPUT_CSV = "./data/LLaVA-Med/MedFlamingo_Localization_D1500_Output_V1.csv"

#PROMPT = """You are a clinical diagnostic assistant. Analyze the provided image for visual evidence of an Adverse Drug Effect (ADE).
#If an ADE is detected, provide the location of the ADE."""


QUESTION = "What is the anatomical location? Respond with only 1–3 words."

PROMPT = f"""
<image>
You are a clinical image assistant.

Look at the image and output ONLY the anatomical location.
- If an ADE is present: output only the location of the ADE.
- If no ADE: output only the body part shown.
- Output must be a short phrase (1–3 words), no explanation.

Examples:
Image: (oral ulcers)
Response: tongue

Image: (swollen leg from drug edema)
Response: left leg

Image: (abdominal rash from ADE)
Response: stomach

Image: (no ADE, only arm visible)
Response: right arm

Image: (swollen leg from drug edema)
Response: right leg

Image: (no ADE, face visible)
Response: face

Question: {QUESTION}
Answer:
"""

def postprocess_location(raw_text: str) -> str:
    # Take first line, strip
    first_line = raw_text.strip().splitlines()[0]

    # Very simple cleanup: keep only letters, spaces, and hyphens
    cleaned = re.sub(r"[^a-zA-Z \-]", " ", first_line).strip()

    # Collapse multiple spaces
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned or first_line.strip()


# ---------- 1. LOAD MODEL FROM HUGGING FACE ----------

def load_med_flamingo():
    """
    Recreate OpenFlamingo-9B architecture and load Med-Flamingo checkpoint
    downloaded from Hugging Face.
    """
    # Build base OpenFlamingo model (same architecture as OpenFlamingo-9B V1)
    model, image_processor, tokenizer = create_model_and_transforms(
        clip_vision_encoder_path="ViT-L-14",      # ✅ valid open_clip model name
        clip_vision_encoder_pretrained="openai",  # use OpenAI CLIP weights
        lang_encoder_path=LLAMA_PATH,
        tokenizer_path=LLAMA_PATH,
        cross_attn_every_n_layers=4,
    )

    # Download checkpoint from Hugging Face
    ckpt_path = hf_hub_download(repo_id=MODEL_REPO, filename=CHECKPOINT_FILENAME)

    ckpt = torch.load(ckpt_path, map_location="cpu")
    # Med-Flamingo checkpoint usually stores model weights under "model_state_dict"
    state_dict = ckpt.get("model_state_dict", ckpt)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Loaded Med-Flamingo checkpoint.")
    if missing:
        print("Missing keys:", len(missing))
    if unexpected:
        print("Unexpected keys:", len(unexpected))

    model.to(DEVICE)
    model.eval()
    return model, image_processor, tokenizer


# ---------- 2. INFERENCE ON A SINGLE IMAGE ----------

@torch.no_grad()
def generate_response_for_image(
    model,
    image_processor,
    tokenizer,
    image_path,
    prompt,
    device="cuda",
    max_new_tokens=8,
):
    """Run Med-Flamingo on a single image and return only the generated answer text."""

    model.eval()

    # ---- Load & prepare image ----
    img = Image.open(image_path).convert("RGB")
    vision = image_processor(images=[img], return_tensors="pt")["pixel_values"]
    vision = vision.unsqueeze(1).unsqueeze(2).to(device)  # (1,1,1,C,H,W)

    # ---- Tokenize prompt ----
    tokens = tokenizer(prompt, return_tensors="pt")
    input_ids = tokens["input_ids"].to(device)
    attn = tokens["attention_mask"].to(device)

    # ---- Generate ----
    with torch.no_grad():
        out = model.generate(
            vision_x=vision,
            lang_x=input_ids,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False,     # greedy → more stable & deterministic
            num_beams=1,
        )

    # ---- Strip prompt tokens ----
    prompt_len = input_ids.shape[1]
    gen_tokens = out[0, prompt_len:]

    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return text


# ----------------------------------------------------------------------

def run_on_folder(
    input_dir: str,
    output_csv: str,
    prompt: str,
    max_new_tokens: int = 64,
):
    """
    Run Med-Flamingo on all images in a folder and save results to CSV.

    CSV columns:
      - image_name
      - raw_response
      - clean_response   (after postprocess_location, if available)
    """

    input_path = Path(input_dir)
    if not input_path.exists():
        raise FileNotFoundError(f"Input folder not found: {input_path.resolve()}")

    # ---------- 1. Load model once ----------
    print("Loading Med-Flamingo...")
    model, image_processor, tokenizer = load_med_flamingo()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    print("Model ready.\n")

    # ---------- 2. Collect image files ----------
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    image_files = sorted(
        [p for p in input_path.iterdir() if p.suffix.lower() in exts]
    )

    print(f"Found {len(image_files)} images.\n")

    # ---------- 3. Open CSV ----------
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image_name", "raw_response", "clean_response"])

        # ---------- 4. Process each image ----------
        for idx, img_path in enumerate(image_files, start=1):
            print(f"[{idx}/{len(image_files)}] Processing {img_path.name}...")
            raw_resp = ""
            clean_resp = ""

            try:
                # ---- 4a. Load image ----
                img = Image.open(img_path).convert("RGB")

                # ---- 4b. Preprocess image ----
                # image_processor is a torchvision.transforms.Compose
                # so we call it directly: image_processor(img)
                img_tensor = image_processor(img)  # shape: (C, H, W)

                if img_tensor.dim() != 3:
                    raise ValueError(
                        f"Expected image tensor of shape (C,H,W), got {img_tensor.shape}"
                    )

                # Build vision_x with shape (B, T_img, F, C, H, W)
                # Here: batch=1, T_img=1, F=1
                vision_x = img_tensor.unsqueeze(0).unsqueeze(1).unsqueeze(2)  # (1,1,1,C,H,W)
                vision_x = vision_x.to(device)

                # ---- 4c. Build text prompt ----
                # Med-Flamingo expects the <image> token before the text
                # (adapt this if your checkpoint uses a slightly different convention)
                flamingo_prompt = f"<image>{prompt.strip()}\nResponse:"

                lang = tokenizer(
                    flamingo_prompt,
                    return_tensors="pt",
                )
                input_ids = lang["input_ids"].to(device)
                attention_mask = lang.get("attention_mask", None)
                if attention_mask is not None:
                    attention_mask = attention_mask.to(device)

                # ---- 4d. Generate ----
                with torch.no_grad():
                    generated_ids = model.generate(
                        vision_x=vision_x,
                        lang_x=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        num_beams=1,
                    )

                # We only keep the new tokens after the prompt
                gen_ids_only = generated_ids[0][input_ids.shape[1]:]
                raw_resp = tokenizer.decode(gen_ids_only, skip_special_tokens=True).strip()

                # ---- 4e. Optional post-processing ----
                try:
                    clean_resp = postprocess_location(raw_resp)
                except NameError:
                    # postprocess_location not defined; just pass raw text
                    clean_resp = raw_resp

                print(f"Response: {clean_resp}\n")

            except Exception as e:
                err_msg = f"Error on {img_path.name}: {e}"
                print(err_msg + "\n")
                raw_resp = err_msg
                clean_resp = ""

            # ---- 4f. Write row ----
            writer.writerow([img_path.name, raw_resp, clean_resp])

    print(f"Saved CSV to: {output_path.resolve()}")






In [ ]:
# ---------- 4. RUN ----------

# Just call this in a Jupyter cell after editing INPUT_DIR / PROMPT above:
# (uncomment to run)

run_on_folder(INPUT_DIR, OUTPUT_CSV, PROMPT, max_new_tokens=8)